# zeon-tubes: Train Detector + Angle Head on Colab GPU

Runs both stages of the two-stage hybrid system end-to-end:

1. **YOLOv11n detector** (cap localization)
2. **MobileNetV3 angle head** (joint→tab direction, 0–360°)

Then aggregates 5-fold cross-validated predictions and prints the final metrics.

**Before running:** upload the `Dataset/` folder (containing `images/` and `annotations.csv`) to your Drive at `MyDrive/zeon-tubes/Dataset/`, and clone this repo into Colab.

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install deps
!git clone https://github.com/ikindacodeabit/zeon_systems.git /content/zeon || (cd /content/zeon && git pull)
%cd /content/zeon
!pip install -q -r requirements.txt

In [ ]:
# Symlink the dataset from Drive into the repo
import os
DATA_SRC = '/content/drive/MyDrive/zeon-tubes/Dataset'
assert os.path.isdir(DATA_SRC), f'Dataset not found at {DATA_SRC}'
if not os.path.exists('Dataset'):
    os.symlink(DATA_SRC, 'Dataset')
!ls Dataset/images | head
!wc -l Dataset/annotations.csv

## Sanity check the eval harness

In [ ]:
!python -m src.evaluate --pred Dataset/annotations.csv --gt Dataset/annotations.csv
# Should print P=R=F1=1.0, mAE=0.

## Classical baseline

In [ ]:
!python -m src.classical --images Dataset/images --out outputs/classical_preds.csv
!python -m src.evaluate --pred outputs/classical_preds.csv --gt Dataset/annotations.csv \
    --collage outputs/classical_worst.png --images-dir Dataset/images

## Stage 1: train YOLOv11n detector (5 folds)

Each fold takes ~5–10 minutes on a Colab T4. Tweak `EPOCHS` if you're short on time.

In [ ]:
EPOCHS = 80
for fold in range(5):
    !python -m src.train_detector --images Dataset/images --ann Dataset/annotations.csv \
        --out outputs/detector_fold{fold}.pt --val-fold {fold} --epochs {EPOCHS} \
        --workdir outputs/yolo_fold{fold}

## Stage 2: train angle head + run end-to-end CV eval

`src/cv_angle.py` trains all 5 angle-head folds and aggregates held-out predictions in one pass.

In [ ]:
!python -m src.cv_angle --images Dataset/images --ann Dataset/annotations.csv \
    --out-csv outputs/hybrid_cv_oracle.csv --epochs 50 --tta
!python -m src.evaluate --pred outputs/hybrid_cv_oracle.csv --gt Dataset/annotations.csv \
    --collage outputs/hybrid_worst.png --images-dir Dataset/images

## End-to-end (YOLO detector + angle head)

Concatenates per-fold detector predictions on each held-out fold and feeds them through the matching angle head.

In [ ]:
import os, csv
from src.dataio import (load_annotations, group_by_image, kfold_image_splits,
                         write_predictions_csv)
from src.infer import run as infer_run

by_image = group_by_image(load_annotations('Dataset/annotations.csv'))
folds = kfold_image_splits(sorted(by_image.keys()), k=5, seed=0)

all_preds = []
for fi, (_, val_imgs) in enumerate(folds):
    # Run inference on val images of fold fi using fold-fi detector + angle head.
    tmp_dir = f'outputs/_fold{fi}_val_imgs'
    os.makedirs(tmp_dir, exist_ok=True)
    for nm in val_imgs:
        src = os.path.abspath(f'Dataset/images/{nm}')
        dst = f'{tmp_dir}/{nm}'
        if not os.path.exists(dst):
            os.symlink(src, dst)
    preds = infer_run(mode='yolo',
                      images_dir=tmp_dir, image_path=None, ann_csv=None,
                      yolo_weights=f'outputs/detector_fold{fi}.pt',
                      angle_weights=f'outputs/angle_cv/angle_fold{fi}.pt',
                      out_csv=None, viz_out=None, tta=True)
    all_preds.extend(preds)

write_predictions_csv('outputs/hybrid_cv_e2e.csv', all_preds)
!python -m src.evaluate --pred outputs/hybrid_cv_e2e.csv --gt Dataset/annotations.csv \
    --collage outputs/hybrid_e2e_worst.png --images-dir Dataset/images

## Demo: single-image inference

In [ ]:
!python -m src.infer --mode yolo --image Dataset/images/2659ffa5-color.png \
    --yolo outputs/detector_fold0.pt --angle outputs/angle_cv/angle_fold0.pt \
    --viz-out outputs/demo_2659ffa5.png
from IPython.display import Image
Image('outputs/demo_2659ffa5.png')

## Save weights back to Drive

In [ ]:
import shutil, os
DST = '/content/drive/MyDrive/zeon-tubes/weights'
os.makedirs(DST, exist_ok=True)
for fi in range(5):
    for fn in [f'outputs/detector_fold{fi}.pt', f'outputs/angle_cv/angle_fold{fi}.pt']:
        if os.path.exists(fn):
            shutil.copy(fn, os.path.join(DST, os.path.basename(fn)))
print('saved weights to', DST)